# U2Net backbone 検証 — v29b と同一パイプラインで backbone のみ RSUネストU-Net に差し替え
- **目的**: 「我々の入力(v28特徴+mdn2ヘッド+同一fold)の上で U2Net アーキは TCN に勝つか / 多様性を足すか」。
- 公開 SCA-U2Net(CV9.55) との違い: 窓512でなく**井戸全長系列** / BatchNorm→**ChanLN**(padding汚染回避) /
  mdn2混合密度ヘッド / 入力は v29 と同じ 192特徴(abs2res+robust scale)。CAM/SAM は最終ステージに付与。
- **flip(反転aug+TTA)は ON** = `b_flip`(単体6.9896 / blend6.9077) と paired 比較すると純粋な backbone 差になる。
- 結果は `artifacts_v29b/results.json` に RUN=`u2net` として合流。最終セルで **GBM+TCN+U2Net の3者ブレンド**も探索。


In [6]:
# ===== 設定 =====
import os, json, math, time, gc, copy, glob, warnings
import joblib
import numpy as np, pandas as pd
from pathlib import Path
import torch, torch.nn as nn, torch.nn.functional as F
from sklearn.model_selection import GroupKFold
warnings.filterwarnings("ignore")

V28_ART = Path(r"C:\Users\kosaka256\Documents\rogii_claude\v28\artifacts_v28")
V29_ART = Path(r"C:\Users\kosaka256\Documents\rogii_claude\v29\artifacts_v29")
PARQUET = V28_ART / "data" / "train.parquet"
DATA    = Path(r"C:\Users\kosaka256\Documents\rogii_claude\rogii-wellbore-geology-prediction")
ART = Path(r"C:\Users\kosaka256\Documents\rogii_claude\v29\artifacts_v29b")   # v29b と共有
(ART / "nn").mkdir(parents=True, exist_ok=True)

RUN = "u2net"
CFG = dict(flip=True, prefix=False, gbmch=False, mtp=False)  # flip=採用済(b_flipとpaired比較)。mtp=多スケール増分補助

N_SPLITS = 5
TOP_K    = 192
FORCE_FEATS = ["md_since", "pf_ancc", "pf_ancc_2", "pf_ancc_3",
               "pf_self", "pf_self_2", "pf_self_3", "pf_nbr", "pf_nbr_2", "pf_nbr_3",
               "pf_self_vs_tw", "pf_self_vs_tw_2", "pf_self_vs_tw_3",
               "pf_nbr_vs_tw", "pf_nbr_vs_tw_2", "pf_nbr_vs_tw_3",
               "pf_self_mean", "pf_nbr_mean", "pf_self_pstd", "pf_nbr_pstd",
               "pf_self_mean_vs_tw", "pf_nbr_mean_vs_tw",
               "pf_agree_std", "pf_agree_std_2", "pf_agree_std_3",
               "pf_nbr_has", "pf_nbr_nndist", "pf_nbr_nrefs", "z_grad"]
# --- U2Net backbone ---
BASE_CH = 64; MID_CH = 16      # 公開版 config(out64/mid16) 準拠
SIDE_W = 0.25                  # deep supervision(側枝mse)の重み
CH_DROP = 0.10
PRE_LEN = 512; PRE_C = 4; CTX_DIM = 64
MTP_OFFS = [4, 16, 64]; MTP_W = 0.1         # MTP: 増分 y[t+k]-y[t] の補助ヘッド
# --- 学習(v29b と同一) ---
HEADS = ["mdn2"]; SEEDS = [0, 1, 2]
EPOCHS = 180; LR = 1e-4; WD = 1e-4; BATCH_WELLS = 256; CROP = 1024
NOISE = 0.02; EMA_DECAY = 0.995; CLIP = 1.0; WARM_FRAC = 0.10
EVAL_TOK = 60_000
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
pp_params = (json.loads((V28_ART / "pp_params.json").read_text())
             if (V28_ART / "pp_params.json").exists() else {"alpha": 1.0, "tau": 85, "w_pf": 0.09})
print(f"RUN={RUN} {CFG} | device={DEVICE} torch={torch.__version__}")


RUN=u2net {'flip': True, 'prefix': False, 'gbmch': False, 'mtp': False} | device=cuda torch=2.11.0+cu128


In [7]:
# ===== 特徴選択(v29 の feat_rank.json を再利用 = TCN と同一特徴) =====
features_all = json.loads((V28_ART / "features.json").read_text())
rank = json.loads((V29_ART / "feat_rank.json").read_text())
rank = [c for c in rank if c != "last_known_tvt"]
feats_nn = [c for c in FORCE_FEATS if c in features_all]
for c in rank:
    if len(feats_nn) >= TOP_K: break
    if c not in feats_nn: feats_nn.append(c)
print("NN特徴:", len(feats_nn), "(v29/v29b と同一)")


NN特徴: 192 (v29/v29b と同一)


In [8]:
# ===== データ読込 + 前処理(v29b と同一) =====
import pyarrow.parquet as pq
need = list(dict.fromkeys(["well", "target", "last_known_tvt", "md_since", "pf_ancc"] + feats_nn))
tbl = pq.read_table(PARQUET, columns=need)
N = tbl.num_rows
g      = tbl.column("well").to_pandas().to_numpy()
y      = tbl.column("target").to_numpy(zero_copy_only=False).astype(np.float64)
lk     = tbl.column("last_known_tvt").to_numpy(zero_copy_only=False).astype(np.float64)
mds    = tbl.column("md_since").to_numpy(zero_copy_only=False).astype(np.float64)
pf_abs = tbl.column("pf_ancc").to_numpy(zero_copy_only=False).astype(np.float64)
X = np.empty((N, len(feats_nn)), np.float32)
for j, c in enumerate(feats_nn):
    X[:, j] = tbl.column(c).to_numpy(zero_copy_only=False).astype(np.float32)
del tbl; gc.collect()

folds = list(GroupKFold(n_splits=N_SPLITS).split(np.zeros(N), y, groups=g))

samp = np.arange(0, N, max(1, N // 120000))
abs2res = []
for j in range(X.shape[1]):
    v = X[samp, j].astype(np.float64); m = np.isfinite(v)
    ok = m.sum() > 100 and np.nanstd(v[m]) > 1e-9
    c_ = float(np.corrcoef(v[m], lk[samp][m])[0, 1]) if ok else 0.0
    a = bool(c_ > 0.98); abs2res.append(a)
    if a: X[:, j] = X[:, j] - lk.astype(np.float32)

med = np.nanmedian(X, axis=0)
q75 = np.nanpercentile(X, 75, axis=0); q25 = np.nanpercentile(X, 25, axis=0)
scl = np.maximum((q75 - q25) / 1.349, 1e-6)
X = (X - med) / scl
X = np.clip(np.nan_to_num(X, nan=0.0, posinf=8.0, neginf=-8.0), -8.0, 8.0).astype(np.float32)

t_mean = float(y.mean()); t_scale = float(y.std())
yn = ((y - t_mean) / t_scale).astype(np.float32)

gbm_oof = np.load(V29_ART / "gbm_oof.npy")
if CFG["gbmch"]:
    X = np.concatenate([X, (((gbm_oof - t_mean) / t_scale)[:, None]).astype(np.float32)], axis=1)

wid_order = pd.unique(g)
well_rows = {}
for w in wid_order:
    idx = np.where(g == w)[0]
    well_rows[w] = idx[np.argsort(mds[idx], kind="stable")]
fold_wells = [(pd.unique(g[tr]), pd.unique(g[va])) for tr, va in folds]

PREF = {}
if CFG["prefix"]:
    pre_fp = ART / "prefix_cache.npz"
    if pre_fp.exists():
        z = np.load(pre_fp, allow_pickle=False)
        PREF = {w: np.nan_to_num(z[w], nan=0.0, posinf=8.0, neginf=-8.0)
                for w in z.files if w in set(wid_order)}
        print("prefix cache load:", len(PREF), "wells")
    else:
        grs = []; raw = {}
        for w in wid_order:
            f = DATA / "train" / f"{w}__horizontal_well.csv"
            if not f.exists(): continue
            df = pd.read_csv(f, usecols=["MD", "Z", "GR", "TVT_input"])
            kn = df.dropna(subset=["MD", "Z", "GR", "TVT_input"]).tail(PRE_LEN)
            if len(kn) < 8: continue
            raw[w] = kn.to_numpy(np.float64); grs.append(raw[w][:, 2])
        allgr = np.concatenate(grs)
        gmed = float(np.nanmedian(allgr))
        giqr = float(max(np.nanpercentile(allgr, 75) - np.nanpercentile(allgr, 25), 1e-6))
        for w, a in raw.items():
            gr_n = np.clip((a[:, 2] - gmed) / giqr, -8, 8)
            tvt_rel = (a[:, 3] - a[-1, 3]) / t_scale
            z_rel = (a[:, 1] - a[-1, 1]) / 50.0
            md_rel = (a[:, 0] - a[-1, 0]) / 1000.0
            PREF[w] = np.nan_to_num(np.stack([gr_n, tvt_rel, z_rel, md_rel], axis=1),
                                    nan=0.0, posinf=8.0, neginf=-8.0).astype(np.float32)
        np.savez_compressed(pre_fp, **PREF)
        print("prefix extracted:", len(PREF), "wells")

K_IN = X.shape[1]
print("rows", N, "| wells", len(wid_order), "| K", K_IN)


rows 3783989 | wells 773 | K 192


In [9]:
# ===== モデル: RSUネストU-Net(1D, ChanLN, CAM/SAM) + 共通ユーティリティ =====
class ChanLN(nn.Module):
    def __init__(self, c):
        super().__init__(); self.ln = nn.LayerNorm(c)
    def forward(self, x):
        return self.ln(x.transpose(1, 2)).transpose(1, 2)

def _up(x, L):
    return x if x.shape[-1] == L else F.interpolate(x, size=L, mode="linear", align_corners=False)

class REBNCONV(nn.Module):
    """Conv1d(k3) + ChanLN + GELU(公開版の BN→LN 置換=padding統計汚染の回避)。"""
    def __init__(self, cin, cout, dil=1):
        super().__init__()
        self.conv = nn.Conv1d(cin, cout, 3, padding=dil, dilation=dil)
        self.ln = ChanLN(cout)
    def forward(self, x):
        return F.gelu(self.ln(self.conv(x)))

class RSU(nn.Module):
    """Residual U-block(高さ height)。pool↓/interp↑、任意長対応(ceil_mode+skip長へ復元)。"""
    def __init__(self, height, cin, mid, cout):
        super().__init__()
        self.height = height
        self.inconv = REBNCONV(cin, cout)
        self.enc = nn.ModuleList([REBNCONV(cout, mid)] +
                                 [REBNCONV(mid, mid) for _ in range(height - 2)])
        self.mid = REBNCONV(mid, mid, dil=2)
        self.dec = nn.ModuleList([REBNCONV(mid * 2, mid) for _ in range(height - 2)])
        self.dec_last = REBNCONV(mid * 2, cout)
        self.pool = nn.MaxPool1d(2, 2, ceil_mode=True)
    def forward(self, x):
        hin = self.inconv(x)
        h = self.enc[0](hin); skips = [h]
        for e in self.enc[1:]:
            h = e(self.pool(h)); skips.append(h)
        h = self.mid(h)
        for i in range(self.height - 2):
            h = self.dec[i](torch.cat([h, skips[-1 - i]], 1))
            h = _up(h, skips[-2 - i].shape[-1])
        h = self.dec_last(torch.cat([h, skips[0]], 1))
        return h + hin

class RSU4F(nn.Module):
    """dilation版(pool無し)ボトルネック用。"""
    def __init__(self, cin, mid, cout):
        super().__init__()
        self.inconv = REBNCONV(cin, cout)
        self.e1 = REBNCONV(cout, mid, 1); self.e2 = REBNCONV(mid, mid, 2); self.e3 = REBNCONV(mid, mid, 4)
        self.m  = REBNCONV(mid, mid, 8)
        self.d3 = REBNCONV(mid * 2, mid, 4); self.d2 = REBNCONV(mid * 2, mid, 2)
        self.d1 = REBNCONV(mid * 2, cout, 1)
    def forward(self, x):
        hin = self.inconv(x)
        h1 = self.e1(hin); h2 = self.e2(h1); h3 = self.e3(h2)
        h = self.m(h3)
        h = self.d3(torch.cat([h, h3], 1))
        h = self.d2(torch.cat([h, h2], 1))
        h = self.d1(torch.cat([h, h1], 1))
        return h + hin

class CAM(nn.Module):
    """channel attention(masked SE)。"""
    def __init__(self, c, r=8):
        super().__init__()
        self.fc1 = nn.Linear(c, c // r); self.fc2 = nn.Linear(c // r, c)
    def forward(self, x, m):
        s = (x * m).sum(-1) / m.sum(-1).clamp(min=1.0)
        w = torch.sigmoid(self.fc2(F.gelu(self.fc1(s))))
        return x * w[:, :, None]

class SAM(nn.Module):
    """spatial attention(chプール→conv→gate)。"""
    def __init__(self, k=7):
        super().__init__()
        self.conv = nn.Conv1d(2, 1, k, padding=k // 2)
    def forward(self, x, m):
        a = x.mean(1, keepdim=True); b = x.amax(1, keepdim=True)
        return x * torch.sigmoid(self.conv(torch.cat([a, b], 1))) * m

class PrefixEnc(nn.Module):
    def __init__(self, c_in, ctx):
        super().__init__()
        self.c1 = nn.Conv1d(c_in, 64, 5, padding=2)
        self.c2 = nn.Conv1d(64, 64, 5, padding=8, dilation=4)
        self.fc = nn.Linear(128, ctx)
    def forward(self, x, m):
        h = F.gelu(self.c1(x * m)); h = F.gelu(self.c2(h)) * m
        s = m.sum(-1)
        mean = h.sum(-1) / s.clamp(min=1.0)
        mx = (h + (m - 1.0) * 1e4).amax(-1)
        mx = torch.where(s > 0, mx, torch.zeros_like(mx))
        return self.fc(torch.cat([mean, mx], dim=-1)) * (s > 0).float()

class U2NetReg(nn.Module):
    """外側U(E1..E5/D4..D1) + 側枝deep supervision + CAM/SAM + mdn2ヘッド。返り値=(out, sides)。"""
    def __init__(self, k_in, head, base, mid, ch_drop, use_ctx=False, pre_c=4, ctx_dim=64, use_mtp=False):
        super().__init__()
        self.head = head; self.use_ctx = use_ctx
        self.inp = nn.Conv1d(k_in, base, 1); self.indrop = nn.Dropout1d(ch_drop)
        if use_ctx:
            self.pre = PrefixEnc(pre_c, ctx_dim); self.ctxp = nn.Linear(ctx_dim, base)
        self.e1 = RSU(7, base, mid, base); self.e2 = RSU(6, base, mid, base)
        self.e3 = RSU(5, base, mid, base); self.e4 = RSU(4, base, mid, base)
        self.e5 = RSU4F(base, mid, base)
        self.d4 = RSU(4, base * 2, mid, base); self.d3 = RSU(5, base * 2, mid, base)
        self.d2 = RSU(6, base * 2, mid, base); self.d1 = RSU(7, base * 2, mid, base)
        self.pool = nn.MaxPool1d(2, 2, ceil_mode=True)
        self.cam = CAM(base); self.sam = SAM(7)
        self.out = nn.Conv1d(base, 5 if head == "mdn2" else 1, 1)
        self.sides = nn.ModuleList([nn.Conv1d(base, 1, 1) for _ in range(4)])  # d2,d3,d4,e5
        self.mtp = nn.Conv1d(base, len(MTP_OFFS), 1) if use_mtp else None
    def forward(self, x, m, px=None, pm=None):
        L = x.shape[-1]
        h = self.inp(self.indrop(x)) * m
        if self.use_ctx and px is not None:
            h = (h + self.ctxp(self.pre(px, pm))[:, :, None]) * m
        h1 = self.e1(h) * m
        h2 = self.e2(self.pool(h1))
        h3 = self.e3(self.pool(h2))
        h4 = self.e4(self.pool(h3))
        h5 = self.e5(self.pool(h4))
        d4 = self.d4(torch.cat([_up(h5, h4.shape[-1]), h4], 1))
        d3 = self.d3(torch.cat([_up(d4, h3.shape[-1]), h3], 1))
        d2 = self.d2(torch.cat([_up(d3, h2.shape[-1]), h2], 1))
        d1 = self.d1(torch.cat([_up(d2, h1.shape[-1]), h1], 1)) * m
        d1 = self.sam(self.cam(d1, m), m)
        out = self.out(d1)
        sides = [_up(sc(t), L) * m for sc, t in zip(self.sides, [d2, d3, d4, h5])]
        mo = self.mtp(d1) if self.mtp is not None else None
        return out, sides, mo

def head_loss(out, yb, mb, head):
    if head == "mse":
        p = out[:, 0]
        return ((p - yb) ** 2 * mb).sum() / mb.sum(), p
    mu1, mu2 = out[:, 0], out[:, 1]
    pl = torch.sigmoid(out[:, 2])
    ls1 = out[:, 3].clamp(-3.0, 2.0); ls2 = out[:, 4].clamp(-3.0, 2.0)
    def logN(mu, ls): return -0.5 * ((yb - mu) / ls.exp()) ** 2 - ls - 0.5 * math.log(2 * math.pi)
    nll = -torch.logsumexp(torch.stack([torch.log(pl + 1e-6) + logN(mu1, ls1),
                                        torch.log(1 - pl + 1e-6) + logN(mu2, ls2)]), dim=0)
    mean = pl * mu1 + (1 - pl) * mu2
    loss = (nll * mb).sum() / mb.sum() + 0.1 * ((mean - yb) ** 2 * mb).sum() / mb.sum()
    return loss, mean

def predict_head(out, head):
    if head == "mse": return out[:, 0]
    pl = torch.sigmoid(out[:, 2]); return pl * out[:, 0] + (1 - pl) * out[:, 1]

class EMA:
    def __init__(self, model, decay):
        self.decay = decay
        self.shadow = {k: v.detach().clone() for k, v in model.state_dict().items()
                       if v.dtype.is_floating_point}
    def update(self, model):
        for k, v in model.state_dict().items():
            if k in self.shadow: self.shadow[k].mul_(self.decay).add_(v.detach(), alpha=1 - self.decay)
    def copy_to(self, model):
        sd = model.state_dict()
        for k, v in self.shadow.items(): sd[k].copy_(v)

def make_batch(wids, train, rng, flip_all=False):
    sel = []
    for w in wids:
        r = well_rows[w]
        if flip_all or (train and CFG["flip"] and rng.random() < 0.5): r = r[::-1]
        if train and len(r) > CROP:
            s = int(rng.integers(0, len(r) - CROP + 1)); r = r[s:s + CROP]
        sel.append((w, r))
    Lm = max(len(r) for _, r in sel); B = len(sel)
    xb = np.zeros((B, Lm, K_IN), np.float32)
    yb = np.zeros((B, Lm), np.float32); mb = np.zeros((B, Lm), np.float32)
    for i, (w, r) in enumerate(sel):
        xb[i, :len(r)] = X[r]; yb[i, :len(r)] = yn[r]; mb[i, :len(r)] = 1.0
    x = torch.from_numpy(xb).to(DEVICE).transpose(1, 2)
    yb = torch.from_numpy(yb).to(DEVICE); mb = torch.from_numpy(mb).to(DEVICE)
    px = pm = None
    if CFG["prefix"]:
        Lp = max(len(PREF.get(w, np.zeros((1, PRE_C), np.float32))) for w, _ in sel)
        pxb = np.zeros((B, Lp, PRE_C), np.float32); pmb = np.zeros((B, Lp), np.float32)
        for i, (w, _) in enumerate(sel):
            a = PREF.get(w)
            if a is not None: pxb[i, :len(a)] = a; pmb[i, :len(a)] = 1.0
        px = torch.from_numpy(pxb).to(DEVICE).transpose(1, 2)
        pm = torch.from_numpy(pmb).to(DEVICE).unsqueeze(1)
    return x, yb, mb, sel, px, pm

@torch.no_grad()
def _predict_once(model, head, wids, flip):
    model.eval(); out = np.zeros(N, np.float64)
    order = sorted(wids, key=lambda w: len(well_rows[w]))
    i = 0
    while i < len(order):
        Lm = len(well_rows[order[i]]); j = i + 1
        while j < len(order):
            Lc = max(Lm, len(well_rows[order[j]]))
            if (j - i + 1) * Lc > EVAL_TOK: break
            Lm = Lc; j += 1
        x, _, mb, sel, px, pm = make_batch(order[i:j], False, None, flip_all=flip)
        with torch.autocast(DEVICE, dtype=torch.bfloat16, enabled=(DEVICE == "cuda")):
            o = model(x, mb.unsqueeze(1), px, pm)[0]
        p = predict_head(o.float(), head).cpu().numpy()
        for b, (w, r) in enumerate(sel): out[r] = p[b, :len(r)]
        i = j
    return out

def predict_wells(model, head, wids):
    p = _predict_once(model, head, wids, False)
    if CFG["flip"]:
        p = 0.5 * (p + _predict_once(model, head, wids, True))
    return p


In [10]:
# ===== 学習: mdn2 × SEEDS × 5fold =====
def cosine_lr(step, total, warm):
    if step < warm: return step / max(1, warm)
    t = (step - warm) / max(1, total - warm)
    return 0.5 * (1.0 + math.cos(math.pi * t))

def new_model(head):
    return U2NetReg(K_IN, head, BASE_CH, MID_CH, CH_DROP,
                    use_ctx=CFG["prefix"], pre_c=PRE_C, ctx_dim=CTX_DIM,
                    use_mtp=CFG.get("mtp", False)).to(DEVICE)

def train_fold(head, seed, k):
    trw, vaw = fold_wells[k]
    fp = ART / "nn" / f"{RUN}_{head}_s{seed}_fold{k}.pt"
    model = new_model(head)
    if fp.exists():
        model.load_state_dict(torch.load(fp, map_location=DEVICE))
        return predict_wells(model, head, list(vaw)), "load"
    torch.manual_seed(1000 * seed + k); rng = np.random.default_rng(1000 * seed + k)
    trw = list(trw); vaw = list(vaw)
    opt = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WD)
    steps_ep = math.ceil(len(trw) / BATCH_WELLS)
    total = steps_ep * EPOCHS; warm = int(total * WARM_FRAC)
    ema = EMA(model, EMA_DECAY); best = (1e18, None); step = 0
    va_idx = np.concatenate([well_rows[w] for w in vaw])
    for ep in range(EPOCHS):
        model.train(); order = rng.permutation(len(trw))
        for i in range(0, len(order), BATCH_WELLS):
            x, yb, mb, _, px, pm = make_batch([trw[j] for j in order[i:i + BATCH_WELLS]], True, rng)
            if NOISE: x = x + NOISE * torch.randn_like(x)
            for pg in opt.param_groups: pg["lr"] = LR * cosine_lr(step, total, warm)
            with torch.autocast(DEVICE, dtype=torch.bfloat16, enabled=(DEVICE == "cuda")):
                out, sides, mo = model(x, mb.unsqueeze(1), px, pm)
            loss, _ = head_loss(out.float(), yb, mb, head)
            if SIDE_W:
                sl = sum(((s[:, 0].float() - yb) ** 2 * mb).sum() / mb.sum() for s in sides) / len(sides)
                loss = loss + SIDE_W * sl
            if CFG.get("mtp", False) and mo is not None:
                for _oi, _kk in enumerate(MTP_OFFS):
                    if yb.shape[1] > _kk:
                        _dt = yb[:, _kk:] - yb[:, :-_kk]
                        _vm = mb[:, _kk:] * mb[:, :-_kk]
                        if _vm.sum() > 0:
                            _pk = mo[:, _oi, :-_kk].float()
                            loss = loss + MTP_W * ((_pk - _dt) ** 2 * _vm).sum() / _vm.sum()
            opt.zero_grad(set_to_none=True); loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), CLIP)
            opt.step(); ema.update(model); step += 1
        bak = copy.deepcopy(model.state_dict()); ema.copy_to(model)
        p = predict_wells(model, head, vaw)
        rm = float(np.sqrt(np.mean((p[va_idx] * t_scale + t_mean - y[va_idx]) ** 2)))
        if not np.isfinite(rm):
            raise RuntimeError(f"val RMSE が非有限 (ep={ep}): 入力/損失にNaN")
        if rm < best[0]: best = (rm, copy.deepcopy(model.state_dict()))
        model.load_state_dict(bak)
    model.load_state_dict(best[1]); torch.save(best[1], fp)
    return predict_wells(model, head, vaw), f"fit best={best[0]:.4f}"

nn_oof = {}
for head in HEADS:
    for seed in SEEDS:
        key = f"{head}_s{seed}"; oof = np.zeros(N, np.float64); t0 = time.time()
        for k in range(N_SPLITS):
            p, tag = train_fold(head, seed, k)
            va = folds[k][1]; oof[va] = p[va]
            gc.collect()
            if DEVICE == "cuda": torch.cuda.empty_cache()
            print(f"[{RUN}] {key} fold{k} [{tag}] "
                  f"rmse={np.sqrt(np.mean((p[va] * t_scale + t_mean - y[va]) ** 2)):.4f}", flush=True)
        nn_oof[key] = oof * t_scale + t_mean
        print(f"== [{RUN}] {key} OOF raw rmse={np.sqrt(np.mean((nn_oof[key] - y) ** 2)):.4f}"
              f" ({time.time() - t0:.0f}s)")
np.savez(ART / f"nn_oof_{RUN}.npz", **nn_oof)


[u2net] mdn2_s0 fold0 [fit best=8.4433] rmse=8.4433
[u2net] mdn2_s0 fold1 [fit best=7.8580] rmse=7.8580


KeyboardInterrupt: 

In [ ]:
# ===== 評価 + blend + results.json 合流 + 3者ブレンド(GBM+TCN+U2Net) =====
pf_res = pf_abs - lk
w_pf = float(pp_params.get("w_pf", 0.09)); tau = pp_params.get("tau", 85)
alpha = float(pp_params.get("alpha", 1.0))
def apply_pp_res(res):
    d = res * (1 - w_pf) + pf_res * w_pf
    if tau: d = d * (1.0 - np.exp(-np.maximum(mds, 0.0) / float(tau)))
    return d * alpha
def raw_rmse(res): return float(np.sqrt(np.mean((res - y) ** 2)))
def overall(res):  return float(np.sqrt(np.mean((apply_pp_res(res) - y) ** 2)))
def wells_over(res, thr=10.0):
    e = apply_pp_res(res) - y
    return int(sum(1 for w in wid_order if np.sqrt(np.mean(e[well_rows[w]] ** 2)) > thr))

ens = np.mean(list(nn_oof.values()), axis=0)
ws = np.arange(0.0, 1.001, 0.02)
scs = [overall((1 - w) * gbm_oof + w * ens) for w in ws]
bi = int(np.argmin(scs)); wb = float(ws[bi])
res_b = (1 - wb) * gbm_oof + wb * ens

rec = dict(cfg={**CFG, "backbone": "u2net"}, crop=CROP, seeds=len(SEEDS),
           nn_raw=raw_rmse(ens), nn_pp=overall(ens), nn_over10=wells_over(ens),
           blend_w=wb, blend_pp=float(scs[bi]), blend_over10=wells_over(res_b))
res_fp = ART / "results.json"
allres = json.loads(res_fp.read_text()) if res_fp.exists() else {}
allres[RUN] = rec
res_fp.write_text(json.dumps(allres, indent=1))
print(f"[{RUN}] NN単体: raw={rec['nn_raw']:.4f} pp={rec['nn_pp']:.4f} >10={rec['nn_over10']}")
print(f"[{RUN}] blend : w={wb:.2f} pp={rec['blend_pp']:.4f} >10={rec['blend_over10']}"
      f"  (GBM単体 {overall(gbm_oof):.4f})")

# ---- 3者ブレンド: GBM + TCN(OTHER_RUN) + U2Net ----
OTHER_RUN = "b_flip"   # 比較相手の TCN(flip採用版)
fp3 = ART / f"nn_oof_{OTHER_RUN}.npz"
if fp3.exists():
    tcn = np.mean(list(np.load(fp3).values()), axis=0)
    b2 = min((overall((1 - w) * gbm_oof + w * tcn), float(w)) for w in ws)
    best3 = None
    for wt in np.arange(0.0, 1.001, 0.05):
        for wu in np.arange(0.0, 1.001 - wt + 1e-9, 0.05):
            wg = 1.0 - wt - wu
            sc = overall(wg * gbm_oof + wt * tcn + wu * ens)
            if best3 is None or sc < best3[0]: best3 = (sc, wg, wt, wu)
    r3 = best3[1] * gbm_oof + best3[2] * tcn + best3[3] * ens
    print(f"\n2way GBM+TCN({OTHER_RUN}) best: w_tcn={b2[1]:.2f} pp={b2[0]:.4f}")
    print(f"3way best: GBM={best3[1]:.2f} TCN={best3[2]:.2f} U2Net={best3[3]:.2f}"
          f" → pp={best3[0]:.4f} >10={wells_over(r3)}  (U2Net追加の上乗せ={best3[0] - b2[0]:+.4f})")
else:
    print(f"({fp3.name} が無いので3者ブレンドはスキップ)")

print("\n=== 全プリセット比較(results.json) ===")
hdr = f"{'RUN':10s} {'nn_raw':>8s} {'nn_pp':>8s} {'>10':>4s} {'w':>5s} {'blend_pp':>9s} {'>10':>4s}  cfg"
print(hdr); print("-" * len(hdr))
for name, r in allres.items():
    on = "+".join(k for k, v in r["cfg"].items() if v is True) or "none"
    bb = r["cfg"].get("backbone", "tcn")
    print(f"{name:10s} {r['nn_raw']:8.4f} {r['nn_pp']:8.4f} {r['nn_over10']:4d}"
          f" {r['blend_w']:5.2f} {r['blend_pp']:9.4f} {r['blend_over10']:4d}  {bb}/{on} crop{r['crop']}")
